## Package Install and Roboflow Datasets

In [ ]:
!pip install ultralytics roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="API-ROBOFLOW")


# Dataset for Ball Detection

project = rf.workspace("computer-vision-project-mjsdu").project("table-tennis-ball-um5tc")
version = project.version(7)
dataset_detection = version.download("yolov11")

# Dataset for Keypoint Detection

project = rf.workspace("computer-vision-project-mjsdu").project("table-tennis-table-net")
version = project.version(12)
dataset_pose = version.download("yolov8")

## Util functions for the Table

In [138]:
import cv2
import numpy as np
import math
from google.colab.patches import cv2_imshow

def euclidean_distance(point1, point2):
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)


def are_points_too_close(point1, point2, threshold=150):
    return euclidean_distance(point1, point2) < threshold


def calculate_homography(src_points, dst_points):
    """
    Compute the homography matrix H.
    src_points: List of source points [(x1, y1), (x2, y2), ...].
    dst_points: List of destination points [(x1, y1), (x2, y2), ...].
    """

    src_points = np.array(src_points, dtype=np.float32)
    dst_points = np.array(dst_points, dtype=np.float32)

    H, _ = cv2.findHomography(src_points, dst_points)
    return H



def calculate_missing_points(points, H):
    """
    Compute the missing point based on the homography matrix H.
    points: List of known points [(x1, y1), (x2, y2), ...].
    H: Homography matrix (3x3).
    """

    points_homogeneous = np.array([[p[0], p[1], 1] for p in points]).T


    transformed_points = H @ points_homogeneous

    transformed_points_cartesian = transformed_points[:2] / transformed_points[2]

    return [(x, y) for x, y in zip(transformed_points_cartesian[0], transformed_points_cartesian[1])]

## Calculate Missing Points + Homography

In [139]:
def calculate_homography(src_points, dst_points):
    """
    Compute the homography matrix H.
    src_points: List of source points [(x1, y1), (x2, y2), ...].
    dst_points: List of destination points [(x1, y1), (x2, y2), ...].
    """

    src_points = np.array(src_points, dtype=np.float32)
    dst_points = np.array(dst_points, dtype=np.float32)

    H, _ = cv2.findHomography(src_points, dst_points)
    return H


def calculate_missing_points(points, H, missing_type, image_width=None, image_height=None):
    try:

        H_inv = np.linalg.inv(H)

        # destination points based on the type of case,
        # tn is top before net point

        if missing_type == "tn":
            if len(points) == 4:  # t_l, t_m, n_l, n_m -> we search for t_r, n_r
                dst_points = [
                    np.array([1, 0, 1]),  # t_r
                    np.array([1, 1, 1])   # n_r
                ]
            else:  # t_m, t_r, n_m, n_r -> we search for t_l, n_l
                dst_points = [
                    np.array([0, 0, 1]),  # t_l
                    np.array([0, 1, 1])   # n_l
                ]

        elif missing_type == "b":  # t_l, t_r, n_l, n_r -> we search for b_l, b_r
            dst_points = [
                np.array([0, 2, 1]),  # b_l
                np.array([1, 2, 1])   # b_r
            ]

        # compute left points
        missing_points = []
        for dst_point in dst_points:
            missing_point = H_inv @ dst_point
            x = missing_point[0] / missing_point[2]
            y = missing_point[1] / missing_point[2]

            x = x * 1.02
            y = y * 1.02

            if image_width is not None and image_height is not None:
                x = np.clip(x, 0, image_width - 1)
                y = np.clip(y, 0, image_height - 1)

            missing_points.append((int(x), int(y)))

        return missing_points

    except Exception as e:
        print(f"Error while finding missing points: {e}")
        return None

## Clip Line

In [140]:
def clip_line_to_frame(p1, p2, frame_shape):
    """
    Clip a line to the edges of the frame.
    :param p1: Tuple (x1, y1) - First point of the line.
    :param p2: Tuple (x2, y2) - Second point of the line.
    :param frame_shape: Tuple (height, width) - Frame dimensions.
    :return: Tuple (p1_clipped, p2_clipped) - Clipped points at the frame edges.
    """

    height, width = frame_shape[:2]

    # check if there is an intersection between points outside the frame
    def intersect(p1, p2, border):
        x1, y1 = p1
        x2, y2 = p2
        if border == 'left':
            x = 0
            y = y1 + (y2 - y1) * (x - x1) / (x2 - x1) if x2 != x1 else y1
        elif border == 'right':
            x = width
            y = y1 + (y2 - y1) * (x - x1) / (x2 - x1) if x2 != x1 else y1
        elif border == 'top':
            y = 0
            x = x1 + (x2 - x1) * (y - y1) / (y2 - y1) if y2 != y1 else x1
        elif border == 'bottom':
            y = height
            x = x1 + (x2 - x1) * (y - y1) / (y2 - y1) if y2 != y1 else x1
        return (int(x), int(y))

    # check if the points are outside the frame
    p1_in_frame = 0 <= p1[0] <= width and 0 <= p1[1] <= height
    p2_in_frame = 0 <= p2[0] <= width and 0 <= p2[1] <= height

    # both points outside the frame, doesn't draw anything
    if not p1_in_frame and not p2_in_frame:
        return None, None


    if not p1_in_frame:
        if p1[0] < 0:
            p1 = intersect(p1, p2, 'left')
        elif p1[0] > width:
            p1 = intersect(p1, p2, 'right')
        if p1[1] < 0:
            p1 = intersect(p1, p2, 'top')
        elif p1[1] > height:
            p1 = intersect(p1, p2, 'bottom')

    if not p2_in_frame:
        if p2[0] < 0:
            p2 = intersect(p2, p1, 'left')
        elif p2[0] > width:
            p2 = intersect(p2, p1, 'right')
        if p2[1] < 0:
            p2 = intersect(p2, p1, 'top')
        elif p2[1] > height:
            p2 = intersect(p2, p1, 'bottom')

    p1 = (int(p1[0]), int(p1[1]))
    p2 = (int(p2[0]), int(p2[1]))
    return p1, p2

## Homography

In [ ]:
def field_homography(frame, table_points):


    # we check if the 2 points are too much close together
    for i, point1 in enumerate(table_points):
        for j, point2 in enumerate(table_points):
            if i != j and (point1 is not None and point2 is not None):
                if euclidean_distance(point1, point2) < 50:
                  table_points[j] = None



    t_l, t_m, t_r, n_l, n_m, n_r = table_points


    # image dimensions
    image_height, image_width = frame.shape[:2]

    dst = [(0, 0), (1, 0), (0, 1), (1, 1)]
    t_r = None
    b_l = None
    b_r = None


    # all 4 points are found
    if t_l and t_r and n_l and n_r:

        H = calculate_homography([t_l, t_r, n_l, n_r], dst)


        missing_points = calculate_missing_points([t_l, t_r, n_l, n_r], H, "b")

        if missing_points:
          b_l, b_r = missing_points
          cv2.circle(frame, (int(b_l[0]), int(b_l[1])), 5, (0, 0, 255), -1)  # draw b_l
          cv2.circle(frame, (int(b_r[0]), int(b_r[1])), 5, (0, 0, 255), -1)  # draw b_r


    # t_l and t_m and n_l and n_m are found but not n_r and t_r
    elif t_l and t_m and n_l and n_m:

        H = calculate_homography([t_l, t_m, n_l, n_m], [(0, 0), (0.5, 0), (0, 1), (0.5, 1)])
        missing_points = calculate_missing_points([t_l, t_m, n_l, n_m], H, "tn")
        if missing_points:
          t_r_, n_r_ = missing_points
        else:
          print("missing points empty")

        # compute n_r
        if t_r is not None and n_r is None:

            n_r = n_r_
            cv2.circle(frame, (int(n_r[0]), int(n_r[1])), 5, (125, 125, 0), -1)  # draw n_r

        # compute t_r
        elif t_r is None and n_r is not None:

            t_r = t_r_
            cv2.circle(frame, (int(t_r[0]), int(t_r[1])), 5, (125, 125, 0), -1)  # draw t_r
        else:
            t_r = t_r_
            n_r = n_r_
            cv2.circle(frame, (int(t_r[0]), int(t_r[1])), 5, (125, 125, 0), -1)  # draw t_r
            cv2.circle(frame, (int(n_r[0]), int(n_r[1])), 5, (125, 125, 0), -1)  # draw n_r

        # compute the points near to the recording player: b_l and b_r (bottom left and bottom right)
        missing_points = calculate_missing_points([t_l, t_r, n_l, n_r], H, "b")

        if missing_points:
          b_l, b_r = missing_points
          cv2.circle(frame, (int(b_l[0]), int(b_l[1])), 5, (0, 0, 255), -1)  # draw b_l
          cv2.circle(frame, (int(b_r[0]), int(b_r[1])), 5, (0, 0, 255), -1)  # draw b_r
        else:
          print("missing points empty")

    # t_m and t_r and n_m and n_r are found but not n_l and t_l
    elif t_m and t_r and n_m and n_r:

        H = calculate_homography([t_m, t_r, n_m, n_r], [(0.5, 0), (1, 0), (0.5, 1), (1, 1)])

        # compute t_l and n_l
        missing_points = calculate_missing_points([t_l, t_m, n_l, n_m], H, "tn")

        if missing_points:
          t_l_, n_l_ = missing_points
        else:
          print("missing points empty")

        if t_l is not None and n_l is None:

            n_l = n_l_
            cv2.circle(frame, (int(n_l[0]), int(n_l[1])), 5, (0, 0, 255), -1)  # draw n_l
        elif t_l is None and n_l is not None:

            t_l = t_l_
            cv2.circle(frame, (int(t_l[0]), int(t_l[1])), 5, (0, 0, 255), -1)  # draw t_l
        else:

            t_l = t_l_
            n_l = n_l_
            cv2.circle(frame, (int(t_l[0]), int(t_l[1])), 5, (0, 0, 255), -1)  # draw t_l
            cv2.circle(frame, (int(n_l[0]), int(n_l[1])), 5, (0, 0, 255), -1)  # draw n_l


        missing_points = calculate_missing_points([t_l, t_r, n_l, n_r], H, "b")
        if missing_points:
          b_l, b_r = missing_points
          cv2.circle(frame, (int(b_l[0]), int(b_l[1])), 5, (0, 0, 255), -1)  # draw b_l
          cv2.circle(frame, (int(b_r[0]), int(b_r[1])), 5, (0, 0, 255), -1)  # draw b_r
        else:
          print("missing points empty")

    elif t_l and t_r and n_l and n_m:

        H = calculate_homography([t_l, t_r, n_l, n_m], [(0, 0), (1, 0), (0, 1), (0.5, 1)])

        missing_points = calculate_missing_points([t_l, t_r, n_l, n_m], H, "tn")
        cv2.circle(frame, (int(n_r[0]), int(n_r[1])), 5, (125, 125, 0), -1)  # draw n_r

        if missing_points:
          _, n_r = missing_points
        else:
          print("missing points empty")

        missing_points = calculate_missing_points([t_l, t_r, n_l, n_r], H, "b")
        if missing_points:
          b_l, b_r = missing_points
          cv2.circle(frame, (int(b_l[0]), int(b_l[1])), 5, (0, 0, 255), -1)  # draw b_l
          cv2.circle(frame, (int(b_r[0]), int(b_r[1])), 5, (0, 0, 255), -1)  # draw b_r
        else:
          print("missing points empty")

    elif t_l and t_r and n_m and n_r:

        H = calculate_homography([t_l, t_r, n_m, n_r], [(0, 0), (1, 0), (0.5, 1), (1, 1)])

        missing_points = calculate_missing_points([t_l, t_r, n_m, n_r], H, "tn")
        cv2.circle(frame, (int(n_l[0]), int(n_l[1])), 5, (0, 0, 255), -1)  # draw n_l

        if missing_points:
          _, n_l = missing_points
        else:
          print("missing points empty")

        missing_points = calculate_missing_points([t_l, t_r, n_l, n_r], H, "b")
        if missing_points:
          b_l, b_r = missing_points
          cv2.circle(frame, (int(b_l[0]), int(b_l[1])), 5, (0, 0, 255), -1)  # draw b_l
          cv2.circle(frame, (int(b_r[0]), int(b_r[1])), 5, (0, 0, 255), -1)  # draw b_r
        else:
          print("missing points empty")

    else:
        # there aren't enough points to compute the homography, so the points will be (0,0), None can give an error
        print("Not enough points to compute the homographya")
        t_l = (0,0)
        t_m = (0,0)
        t_r = (0,0)
        n_l = (0,0)
        n_m = (0,0)
        n_r = (0,0)
        b_l = (0,0)
        b_r = (0,0)



    table_points = [t_l, t_m, t_r, n_l, n_m, n_r]

    ################
    # draw circles #
    ################

    for point in table_points:
        if point :
            cv2.circle(frame, (int(point[0]), int(point[1])), 5, (0, 255, 0), -1)

    b_l = (int(b_l[0]), int(b_l[1]))
    b_r = (int(b_r[0]), int(b_r[1]))

    ##############
    # draw lines #
    ##############

    # lines between points are drawn to connect the edges, if the point is outside the image
    # the line will still be drawn

    p1, p2 = clip_line_to_frame(t_l, t_r, frame.shape)
    cv2.line(frame, p1, p2, (0, 255, 0), 2)

    p1, p2 = clip_line_to_frame(t_l, b_l, frame.shape)
    cv2.line(frame, p1, p2, (0, 255, 0), 2)

    p1, p2 = clip_line_to_frame(t_r, b_r, frame.shape)
    cv2.line(frame, p1, p2, (0, 255, 0), 2)

    p1, p2 = clip_line_to_frame(b_l, b_r, frame.shape)
    cv2.line(frame, p1, p2, (0, 255, 0), 2)

    table_points = [t_l, t_r, b_r, b_l]

    return frame, table_points

## Draw 2D top-view field


In [141]:
import cv2
import numpy as np

def draw_ping_pong_table(frame):
    """
    Draw the table tennis field from a top-view on the top right corner of the image.
    """

    TABLE_WIDTH = 152   # short edge of the table
    TABLE_HEIGHT = 274   # long edge of the table
    TABLE_MIDDLE = TABLE_HEIGHT // 2

    # offsets for the table
    TABLE_OFFSET_X = 10
    TABLE_OFFSET_Y = 10

    # colors (BGR)
    BACKGROUND_COLOR = (255, 0, 0)  # blu
    TABLE_COLOR = (255, 255, 255)  # white
    NET_COLOR = (0, 0, 0)  # black

    frame_height, frame_width = frame.shape[:2]

    # area to draw the 2D field
    table_area = frame[TABLE_OFFSET_Y:TABLE_OFFSET_Y + TABLE_HEIGHT,
                      frame_width - TABLE_WIDTH - TABLE_OFFSET_X:frame_width - TABLE_OFFSET_X]
    table_area[:] = BACKGROUND_COLOR

    cv2.rectangle(frame,
                  (frame_width - TABLE_WIDTH - TABLE_OFFSET_X, TABLE_OFFSET_Y),
                  (frame_width - TABLE_OFFSET_X, TABLE_OFFSET_Y + TABLE_HEIGHT),
                  TABLE_COLOR, 4)

    middle_y = TABLE_OFFSET_Y
    middle_start = (frame_width - TABLE_WIDTH // 2 - TABLE_OFFSET_X, middle_y)
    middle_end = (frame_width - TABLE_WIDTH // 2 - TABLE_OFFSET_X, middle_y + TABLE_HEIGHT)
    cv2.line(frame, middle_start, middle_end, TABLE_COLOR, 3)

    net_y = TABLE_OFFSET_Y + TABLE_HEIGHT // 2
    net_start = (frame_width - TABLE_WIDTH - TABLE_OFFSET_X, net_y)
    net_end = (frame_width - TABLE_OFFSET_X, net_y)
    cv2.line(frame, net_start, net_end, NET_COLOR, 3)

    return frame


## Ball Projection on 2D top-view

In [144]:
import cv2
import numpy as np

def estimate_ball_position_in_top_view(table_corners, ball_center, table_dims):
    """
    Transforms the ball's position from image coordinates to a top-down 2D space.
    """
    TABLE_WIDTH, TABLE_HEIGHT = table_dims

    # convert table corners to numpy arrays
    table_corners = np.array(table_corners, dtype=np.float32)

    # define the top-down rectangle coordinates
    top_view_corners = np.array([
        [0, 0],  # Top-left
        [TABLE_WIDTH, 0],  # Top-right
        [TABLE_WIDTH, TABLE_HEIGHT],  # Bottom-right
        [0, TABLE_HEIGHT]  # Bottom-left
    ], dtype=np.float32)

    H, _ = cv2.findHomography(table_corners, top_view_corners)

    # we check if the homography matrix is computed
    if H is None:
        raise ValueError("Failed to compute homography. Ensure table_corners and top_view_corners are valid.")


    x_center = ball_bbox[0] + ball_bbox[2] / 2
    y_center = ball_bbox[1] + ball_bbox[3] / 2
    ball_center = [x_center, y_center]

    ball_center = np.array([[ball_center]], dtype=np.float32)


    # ball center to top-view
    try:
        ball_position_top_view = cv2.perspectiveTransform(ball_center, H)[0][0]
    except Exception as e:
        print("Error during perspectiveTransform:", e)
        return None

    return ball_position_top_view


def draw_ping_pong_table_with_ball(frame, ball_position_top_view, table_dims):
    """
    Draws a ping pong table in the top-right corner of the frame and overlays the ball's position.
    """
    TABLE_WIDTH, TABLE_HEIGHT = table_dims
    TABLE_MIDDLE = TABLE_HEIGHT // 2
    # measurement used to adjust the position error
    TABLE_WIDTH_PLUS = TABLE_WIDTH // 8
    TABLE_HEIGHT_PLUS = TABLE_HEIGHT // 8

    TABLE_OFFSET_X = 10
    TABLE_OFFSET_Y = 10


    frame_height, frame_width = frame.shape[:2]

    table_area_start_x = frame_width - TABLE_WIDTH - TABLE_OFFSET_X
    table_area_end_x = frame_width - TABLE_OFFSET_X
    table_area_start_y = TABLE_OFFSET_Y
    table_area_end_y = TABLE_OFFSET_Y + TABLE_HEIGHT


    table_area = frame[table_area_start_y:table_area_end_y,
                       table_area_start_x:table_area_end_x]
    table_area[:] = (255, 0, 0)


    cv2.rectangle(frame,
                  (table_area_start_x, table_area_start_y),
                  (table_area_end_x, table_area_end_y),
                  (255, 255, 255), 4)


    middle_line_start = (table_area_start_x + TABLE_WIDTH // 2, table_area_start_y)
    middle_line_end = (table_area_start_x + TABLE_WIDTH // 2, table_area_end_y)
    cv2.line(frame, middle_line_start, middle_line_end, (255, 255, 255), 3)

    net_line_start = (table_area_start_x, table_area_start_y + TABLE_MIDDLE)
    net_line_end = (table_area_end_x, table_area_start_y + TABLE_MIDDLE)
    cv2.line(frame, net_line_start, net_line_end, (0, 0, 0), 3)

    # if ball position is found we draw it in the overlay area
    if ball_position_top_view is not None:
        # scale the ball position to fit the overlay area
        x_scaled = int(((ball_position_top_view[0] - TABLE_WIDTH_PLUS ) / TABLE_WIDTH) * TABLE_WIDTH) + table_area_start_x
        y_scaled = int(((ball_position_top_view[1] - TABLE_HEIGHT_PLUS) / TABLE_HEIGHT) * TABLE_HEIGHT) + table_area_start_y

        # draw the ball as a red circle
        cv2.circle(frame, (x_scaled, y_scaled), 5, (0, 0, 255), -1)

    return frame

## Main - Takes a video as input

In [ ]:
import cv2
import time
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# load both yolo models
model_pose = YOLO('/content/yolov8l_pose.pt')
model_detect = YOLO('/content/yolov11_detection.pt')

video_path = '/content/video.mp4'
output_path = "/content/output_video.mp4"

cap = cv2.VideoCapture(video_path)

# video info
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))



fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))


frame_count = 0
start_time = time.time()

TABLE_DIM = (152, 274)
BALL_DIM = 40


while cap.isOpened():
      ret, frame = cap.read()
      if not ret:
          break

      ##################
      # BALL DETECTION #
      ##################

      # use yovo11 to detect the ball
      result = model_detect.predict(frame)


      ball_bbox = []
      if len(result[0].boxes) > 0:

        ball_bbox = result[0].boxes.xyxy[0].cpu().numpy()

        x1, y1, x2, y2 = map(int, ball_bbox)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

      ###################
      # TABLE DETECTION #
      ###################

      # use YOLOv8 to predict the table keypoints
      results = model_pose.predict(frame)

      result = results[0]

      keypoints = result.keypoints if hasattr(result, 'keypoints') else None

      table_points = []

      processed_frame = frame

      if keypoints is not None:

          keypoints = keypoints.data.cpu().numpy()

          if keypoints.shape[1] > 3:


              for i in range(keypoints.shape[1]):
                  x, y = int(keypoints[0][i][0]), int(keypoints[0][i][1])
                  table_points.append((x, y))

              processed_frame, table_points = field_homography(frame, table_points)


          else:
              # no keypoints in the original frame, use the original one
              processed_frame = frame
      else:
          # no keypoints in the original frame, use the original one
          processed_frame = frame

      if len(table_points) == 4 and len(ball_bbox) > 0:

        ball_position = estimate_ball_position_in_top_view(table_points, ball_bbox, TABLE_DIM)

        processed_frame = draw_ping_pong_table_with_ball(processed_frame, ball_position, TABLE_DIM)

      else:
        processed_frame = draw_ping_pong_table(processed_frame)

      out.write(processed_frame)


cap.release()
out.release()
cv2.destroyAllWindows()

end_time = time.time()
print(f"Total time: {end_time - start_time:.2f} seconds")
print(f"Frame processed: {frame_count}")